In [ ]:
from pathlib import Path
import time
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay
from pynq.lib.video import VideoMode

WIDTH, HEIGHT, BPP = 640, 480, 24
videoMode = VideoMode(WIDTH, HEIGHT, BPP)

bit_candidates = [
    Path('/home/xilinx/wave_laplacian.bit'),
    Path('/home/xilinx/base_wrapper.bit'),
    Path('/home/xilinx/base.bit'),
]
BIT_PATH = next((str(p) for p in bit_candidates if p.exists()), None)
if BIT_PATH is None:
    raise FileNotFoundError('No expected bitstream found in /home/xilinx')

overlay = Overlay(BIT_PATH)
print('Loaded:', BIT_PATH)
print('IPs found:')
for name in overlay.ip_dict.keys():
    print(' -', name)


In [ ]:
pixgen = overlay.pixel_generator_0
imgen_vdma = overlay.video.axi_vdma_0.readchannel
hdmi_out = overlay.video.hdmi_out
hdmi_out._vdma = overlay.video.axi_vdma

try:
    imgen_vdma.stop()
except Exception:
    pass
try:
    hdmi_out.close()
except Exception:
    pass

imgen_vdma.mode = videoMode
imgen_vdma.start()
time.sleep(0.5)

hdmi_out.configure(videoMode)
hdmi_out.start()

print('VDMA running:', imgen_vdma.running)
print('HDMI started.')
print('MMIO registers:', list(pixgen.register_map._instances.keys()))


In [ ]:
def wait_frame(timeout=2.0):
    start = time.time()
    while time.time() - start < timeout:
        if imgen_vdma._mmio.read(0x34) & 0x1000:
            return True
        time.sleep(0.02)
    return False

def get_frame(show=False, title='Wave Output'):
    if not wait_frame():
        print('No frame ready.')
        return None
    frame = imgen_vdma.readframe()
    hdmi_out.writeframe(frame)
    if show:
        plt.figure(figsize=(8, 6))
        plt.imshow(frame)
        plt.title(title)
        plt.axis('off')
        plt.show()
    return frame

frame = get_frame(show=True, title='Wave Output Preview')
if frame is not None:
    print('Frame shape:', frame.shape)


In [ ]:
# Refresh a few frames and keep HDMI updated.
FRAMES = 30
DELAY = 0.03

for i in range(FRAMES):
    frame = get_frame(show=False)
    if frame is None:
        print('Stopped at frame', i)
        break
    time.sleep(DELAY)

print('Refresh loop done.')


In [ ]:
# Your current HDL does not consume gp0..gp7 yet, so these writes are for inspection only.
pixgen.register_map


In [ ]:
save_path = '/home/xilinx/wave_output_preview.png'
frame = get_frame(show=False)
if frame is not None:
    plt.imsave(save_path, np.array(frame))
    print('Saved to', save_path)
else:
    print('No frame to save.')


In [ ]:
try:
    hdmi_out.close()
except Exception:
    pass
try:
    imgen_vdma.stop()
except Exception:
    pass
print('Shutdown ok.')
